# Contextual Belief Updating — MVP Notebook

This notebook runs the full experiment from the NeurIPS submission:
> **"Learning Compact Market-State Representations from Prediction Markets via Contextual Belief Updating"**

## What this does

1. Loads the `CanonicalDataset` from the frozen parquet cache
2. Builds a `BeliefUpdatingManifest` — training examples of the form `(stale_features, context_matrix, label)`
3. Runs the **4-rung evaluation ladder**:
   - **Rung 1** — stale-only (baseline: outdated local snapshot)
   - **Rung 2** — stale + raw context (aggregated mean/max of family siblings)
   - **Rung 3** — stale + compact embedding `z_t` (DeepSets encoder)
   - **Rung 4** — stale + corrupted `z̃_t` (shuffled context control)
4. Reports the compression claim: does rung 3 ≈ rung 2 >> rung 1?

## Setup

Activate your virtualenv and install dependencies before running:
```bash
conda activate polymarket          # or: source .venv/bin/activate
pip install -e .                   # install polymarket_research package
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from polymarket_research.data.canonical import CanonicalDataset
from polymarket_research.utils import setup_root

from polymarket_research.belief_updating import (
    BeliefUpdatingDatasetBuilder,
    BeliefUpdatingSpec,
    BeliefUpdatingMVPExperiment,
    BeliefUpdatingMVPConfig,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)


## 1. Load the Canonical Dataset

Use the same parquet cache as all other frozen notebooks.

In [ ]:
REPO_ROOT = setup_root()
CANONICAL_CACHE_DIR = REPO_ROOT / "frozen_notebooks" / "running_artefacts" / "canonical_dataset"
MANIFEST_CACHE_DIR  = REPO_ROOT / "frozen_notebooks" / "running_artefacts" / "belief_updating_manifest"

# Load canonical (same pattern as other notebooks)
if (CANONICAL_CACHE_DIR / "markets.parquet").exists():
    canonical = CanonicalDataset.from_parquet(CANONICAL_CACHE_DIR)
    print("Loaded canonical from parquet cache:", CANONICAL_CACHE_DIR)
else:
    raise FileNotFoundError(
        f"Canonical cache not found at {CANONICAL_CACHE_DIR}.\n"
        "Run frozen_notebooks/0_data_layers.ipynb first to build it."
    )

canonical.summary()

In [ ]:
# Quick look at the market table
canonical.markets[['market_id', 'question', 'domain', 'family_id', 'final_outcome', 'probability_rows']].head(5)

## 2. Build the Belief Updating Dataset

For each market the builder creates examples at multiple (horizon, delta) combinations:
- **horizon_hours** = time before resolution at which we observe the context (24h, 72h, 168h)
- **delta_hours** = how stale the target's local snapshot is (24h, 72h, 168h before context_time)

Each example: stale features at `context_time - delta_hours` + family siblings at `context_time` → label

In [ ]:
spec = BeliefUpdatingSpec(
    horizons_hours=(24, 72, 168),
    delta_hours_options=(24, 72, 168),
    max_snapshot_staleness_hours=12.0,
    min_family_size=1,               # require at least 1 sibling
)

# Try loading from cache, or rebuild
from polymarket_research.belief_updating import BeliefUpdatingManifest
import os

if (MANIFEST_CACHE_DIR / "examples.parquet").exists():
    manifest = BeliefUpdatingManifest.from_parquet(MANIFEST_CACHE_DIR, spec=spec)
    print("Loaded manifest from cache:", MANIFEST_CACHE_DIR)
else:
    builder  = BeliefUpdatingDatasetBuilder(canonical=canonical, spec=spec)
    manifest = builder.build(show_progress=True)
    manifest.save(MANIFEST_CACHE_DIR)
    print("Built and saved manifest to:", MANIFEST_CACHE_DIR)

manifest.summary()

In [ ]:
# Examples table: stale features + aggregated context + label
examples = manifest.examples
print("Shape:", examples.shape)
print("Columns:", list(examples.columns))
examples[['market_id', 'horizon_hours', 'delta_hours_int', 'stale_yes_probability', 'n_siblings', 'label']].head(8)

In [ ]:
# Distribution of examples by (horizon, delta)
(
    examples
    .groupby(['horizon_hours', 'delta_hours_int'])
    .agg(n_examples=('label', 'count'), pct_yes=('label', 'mean'))
    .reset_index()
)

In [ ]:
# Distribution of family size (n_siblings)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

examples['n_siblings'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#4c72b0')
axes[0].set_title('Family siblings per example')
axes[0].set_xlabel('n_siblings')

examples['delta_hours_int'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#55a868')
axes[1].set_title('Staleness gap (delta_hours)')
axes[1].set_xlabel('delta_hours')

fig.tight_layout()
plt.show()

## 3. Run the Experiment Ladder

This runs all four rungs in order. Total runtime depends on dataset size and device:
- Rungs 1–2 (sklearn GBM): fast (~seconds)
- Rungs 3–4 (PyTorch DeepSets): ~1–5 min on CPU; use `device='cuda'` or `device='mps'` to speed up

**Tip**: reduce `n_epochs` for a quick first run; increase to 30–50 for final results.

In [ ]:
import torch

# Auto-detect device
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Using device: {device}")

config = BeliefUpdatingMVPConfig(
    test_fraction=0.25,
    n_epochs=30,           # ← increase for final results
    batch_size=128,
    learning_rate=1e-3,
    encoder_hidden_dim=64,
    encoder_output_dim=32, # d_z
    max_context_size=16,
    device=device,
    random_state=42,
)

ladder = BeliefUpdatingMVPExperiment(manifest=manifest, config=config)
results = ladder.run(verbose=True)


## 4. Results

In [ ]:
# Per-rung metrics table
rung_df = results.to_dataframe()
rung_df

In [ ]:
# The compression claim: retention = how much of the raw-context gain is kept by the embedding
results.compression_summary()

In [ ]:
# Visualize log-loss across rungs
rung_order = ['stale_only', 'stale_plus_raw', 'stale_plus_embedding', 'stale_plus_corrupted']
rung_labels = [
    'Stale-only\n(rung 1)',
    'Stale + Raw\n(rung 2)',
    'Stale + Embed\n(rung 3)',
    'Stale + Corrupt\n(rung 4)',
]

plot_df = rung_df.set_index('rung').loc[rung_order]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric, title in zip(axes, ['log_loss', 'brier', 'roc_auc'], ['Log-Loss ↓', 'Brier ↓', 'ROC-AUC ↑']):
    colors = ['#4c72b0', '#55a868', '#c44e52', '#8172b2']
    bars = ax.bar(rung_labels, plot_df[metric].values, color=colors)
    for bar, val in zip(bars, plot_df[metric].values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylim(bottom=0)

fig.suptitle('Experiment Ladder: 4-Rung Evaluation')
fig.tight_layout()
plt.show()

## 5. Hard Slices

The paper calls for special evaluation on examples where the representation should be most valuable:
- **Long staleness**: delta_hours > 72
- **Uncertain stale prior**: stale_yes_probability ∈ [0.35, 0.65]
- **Large family**: n_siblings ≥ 3

In [ ]:
# Show slice sizes
ex = manifest.examples
slices = {
    'all': ex,
    'long_staleness (delta≥72h)': ex[ex['delta_hours_int'] >= 72],
    'uncertain_prior (|p-0.5|<0.15)': ex[ex['stale_confidence_margin'] < 0.15],
    'rich_family (siblings≥3)': ex[ex['n_siblings'] >= 3],
}

pd.DataFrame([
    {'slice': name, 'n_examples': len(df), 'pct_yes': df['label'].mean()}
    for name, df in slices.items()
])

## 6. Next Steps

Once this MVP produces clean results, the natural extensions are:

1. **Semantic neighbors** — enrich context beyond family siblings using question embeddings (sentence-transformers)
2. **External covariates** — add BTC/ETH shock features as a global context row
3. **Transfer probes** — freeze the encoder and evaluate on trust / repricing benchmarks  
4. **Staleness-stratified plots** — plot compression retention vs. delta_hours to show where the representation adds most value
5. **Set Transformer** — upgrade the set encoder from DeepSets to a Set Transformer with attention